# Comparing Krylov methods and Steepest Descent

This notebook compares the **current modular implementations** of Conjugate Gradient
(CG), GMRES, BiCGSTAB, symmetric Lanczos, and **Steepest Descent** (a gradient-based
baseline, rather than a general-purpose Krylov solver).

We examine three complementary questions:

1. **SPD / Hermitian PD systems:** How do Steepest Descent, CG, GMRES and BiCGSTAB compare?
2. **Nonsymmetric systems:** How do GMRES and BiCGSTAB compare, including restarted GMRES?
3. **Lanczos projections:** How closely do Ritz values approximate eigenvalues, and how
   accurately does a Galerkin projection solve a symmetric linear system?

All problems use a known solution $x_*$, so we can report both the true residual
$\|b-Ax_k\|_2$ and the relative solution error $\|x_k-x_*\|_2/\|x_*\|_2$.
Click **Run comparison** after adjusting the controls. The displayed work is
recomputed each time; no cells rely on stale plots or manually copied results.

In [2]:
from pathlib import Path
import sys
import time

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "gradient_methods").is_dir() and (candidate / "krylov_methods").is_dir():
        repo_root = candidate
        break
else:
    raise RuntimeError("Start Jupyter inside your repository checkout.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

from gradient_methods.steepest_descent_solver import solve_steepest_descent
from krylov_methods.conjugate_gradient_solver import solve_conjugate_gradient
from krylov_methods.gmres_solver.GMRES import gmres
from krylov_methods.bicgstab_solver import solve_bicgstab
from krylov_methods.symmetric_lanczos_solver import lanczos_tridiagonalize, lanczos_galerkin
from krylov_methods.notebooks.notebook_utils import (
    spd_problem, hermitian_pd_problem, nonsymmetric_problem,
    residual_plot, error_plot, ritz_spectrum_plot, ritz_residual_plot,
)

print("Repository:", repo_root)
print("NumPy:", np.__version__)

Repository: /home/amal-ma/Downloads/alg
NumPy: 2.2.4


## 1. Symmetric positive-definite and complex Hermitian PD systems

Steepest Descent and CG require positive-definite structure; GMRES and BiCGSTAB
also apply but use different recurrences. We generate $A=Q\Lambda Q^H$ with
geometrically spaced eigenvalues and prescribed condition number
$\kappa_2(A)=\lambda_{\max}/\lambda_{\min}$.

**What to inspect:** number of iterations, runtime, final true residual,
and relative solution error. A small residual does not guarantee equally
small forward error when the system is ill-conditioned.

All four solvers use the same **absolute-plus-relative** stopping rule:

<p align="center">$\displaystyle \|b-Ax_k\|_2\leq\max(10^{-12},\mathrm{rtol}\,\|b\|_2).$</p>

Unpreconditioned runs make the algorithms directly comparable. For specific
preconditioners, use the individual method notebooks.

In [4]:
spd_kind = widgets.Dropdown(
    options=[("Real SPD", "real"), ("Complex Hermitian PD", "complex")],
    description="System:")
spd_n = widgets.IntSlider(value=20, min=5, max=45, step=5, description="Dimension:")
spd_cond = widgets.Dropdown(options=[("10", 10.), ("100", 100.),
                                         ("1,000", 1000.)], value=100., description="Condition:")
spd_rtol = widgets.Dropdown(options=[("1e-6", 1e-6), ("1e-8", 1e-8),
                                          ("1e-10", 1e-10)], value=1e-8, description="rtol:")
spd_budget = widgets.IntSlider(value=3000, min=100, max=8000, step=100,
                               description="SD budget:")
spd_run = widgets.Button(description="Run comparison", button_style="primary")
spd_out = widgets.Output()

def run_spd(_=None):
    with spd_out:
        clear_output(wait=True)
        generator = hermitian_pd_problem if spd_kind.value == "complex" else spd_problem
        A, b, x_true = generator(n=spd_n.value, condition=spd_cond.value, seed=42)
        rtol, atol = spd_rtol.value, 1e-12
        threshold = max(atol, rtol * np.linalg.norm(b))
        print(f"Dimension={len(b)} | dtype={A.dtype} | condition={np.linalg.cond(A):.3e}")
        print(f"Hermitian error={np.linalg.norm(A-A.conj().T):.2e}")
        print(f"||b||={np.linalg.norm(b):.3e} | stopping threshold={threshold:.3e}")
        gmres_points = [(0, float(np.linalg.norm(b)))]
        methods = {
            "Steepest Descent": lambda: solve_steepest_descent(
                A, b, rtol=rtol, atol=atol, max_iter=spd_budget.value),
            "CG": lambda: solve_conjugate_gradient(
                A, b, rtol=rtol, atol=atol, max_iter=10*len(b)),
            "GMRES (unrestarted)": lambda: gmres(
                A, b, rtol=rtol, atol=atol, maxiter=len(b),
                callback=lambda i, true_norm, estimate: gmres_points.append((i, true_norm))),
            "BiCGSTAB": lambda: solve_bicgstab(
                A, b, rtol=rtol, atol=atol, max_iter=10*len(b)),
        }
        histories = {}
        for label, run in methods.items():
            started = time.perf_counter()
            result = run()
            elapsed = time.perf_counter()-started
            x = result.x if label.startswith("GMRES") else result.solution
            res = np.linalg.norm(b-A@x)
            err = np.linalg.norm(x-x_true)/np.linalg.norm(x_true)
            hist = (tuple(np.array(gmres_points).T) if label.startswith("GMRES")
                    else result.residual_norms)
            histories[label] = hist
            print(f"{label:22} converged={str(result.converged):5} "
                  f"iter={result.iterations:5} residual={res:.2e} "
                  f"rel_error={err:.2e} time={elapsed:.3f}s")
        residual_plot(histories, threshold, f"True residuals — {spd_kind.label}")
        print("Note: GMRES callback points use their true Arnoldi iteration indices.")

spd_run.on_click(run_spd)
display(widgets.VBox([spd_kind, spd_n, spd_cond, spd_rtol, spd_budget, spd_run, spd_out]))

## 2. Nonsymmetric systems: restarted and unrestarted GMRES vs BiCGSTAB

CG and Steepest Descent are **not applicable to arbitrary nonsymmetric matrices**.
Here we compare GMRES with and without restart to BiCGSTAB on the same
reproducible, diagonally dominant nonsymmetric system.

- Unrestarted GMRES minimizes the true Euclidean residual at each Arnoldi step.
- Restarting bounds the basis size but can slow or stall convergence.
- BiCGSTAB has short recurrences and may have nonmonotone residuals.

The plots show **true residual norms**, not left-preconditioned estimates.
For GMRES, intermediate true norms are recorded through the callback.

In [6]:
ns_n = widgets.IntSlider(value=30, min=10, max=80, step=10, description="Dimension:")
ns_restart = widgets.IntSlider(value=10, min=3, max=30, step=1, description="Restart:")
ns_budget = widgets.IntSlider(value=200, min=30, max=500, step=10, description="Max steps:")
ns_rtol = widgets.Dropdown(options=[("1e-6", 1e-6), ("1e-8", 1e-8),
                                         ("1e-10", 1e-10)], value=1e-8, description="rtol:")
ns_run = widgets.Button(description="Run comparison", button_style="primary")
ns_out = widgets.Output()

def run_ns(_=None):
    with ns_out:
        clear_output(wait=True)
        A, b, x_true = nonsymmetric_problem(n=ns_n.value, seed=42)
        threshold = max(1e-12, ns_rtol.value * np.linalg.norm(b))
        print(f"Dimension={len(b)} | ||A-A^T||={np.linalg.norm(A-A.T):.3e}")
        print(f"Condition number={np.linalg.cond(A):.3e} | threshold={threshold:.3e}")
        full_points = [(0, float(np.linalg.norm(b)))]
        restarted_points = [(0, float(np.linalg.norm(b)))]
        methods = {
            "GMRES (full)": lambda: gmres(A, b, restart=None, maxiter=len(b),
                rtol=ns_rtol.value, atol=1e-12,
                callback=lambda i, true_norm, est: full_points.append((i, true_norm))),
            "GMRES (restarted)": lambda: gmres(A, b, restart=min(ns_restart.value, len(b)),
                maxiter=ns_budget.value, rtol=ns_rtol.value, atol=1e-12,
                callback=lambda i, true_norm, est: restarted_points.append((i, true_norm))),
            "BiCGSTAB": lambda: solve_bicgstab(A, b, max_iter=ns_budget.value,
                rtol=ns_rtol.value, atol=1e-12),
        }
        histories = {}
        for label, run in methods.items():
            started = time.perf_counter()
            result = run()
            elapsed = time.perf_counter()-started
            x = result.x if label.startswith("GMRES") else result.solution
            history = (tuple(np.array(full_points).T) if label == "GMRES (full)"
                       else tuple(np.array(restarted_points).T) if label == "GMRES (restarted)"
                       else result.residual_norms)
            histories[label] = history
            error = np.linalg.norm(x-x_true)/np.linalg.norm(x_true)
            cycles = f", cycles={result.cycles}" if label.startswith("GMRES") else ""
            print(f"{label:20} converged={str(result.converged):5} "
                  f"iter={result.iterations:4}{cycles} "
                  f"residual={np.linalg.norm(b-A@x):.2e} "
                  f"rel_error={error:.2e} time={elapsed:.3f}s")
        residual_plot(histories, threshold, "Nonsymmetric-system convergence")

ns_run.on_click(run_ns)
display(widgets.VBox([ns_n, ns_restart, ns_budget, ns_rtol, ns_run, ns_out]))

## 3. Lanczos: spectrum and projected linear solve

Symmetric Lanczos constructs an orthonormal basis $V_m$ and a tridiagonal
projection $T_m=V_m^HAV_m$. Unlike CG, it is **primarily a basis / spectral
method**, although the Galerkin approximation

<p align="center">$\displaystyle x_m=x_0+V_my_m,\qquad T_my_m=\|b-Ax_0\|_2e_1$</p>

can solve a linear system when $T_m$ is nonsingular. We compare Ritz values
against the exact spectrum, check orthogonality, and test the Galerkin
solution against a direct solve. Partial-dimensional projections are not
expected to solve the full system to a tight tolerance.

In [8]:
lan_n = widgets.IntSlider(value=20, min=5, max=50, step=5, description="Dimension:")
lan_m = widgets.IntSlider(value=10, min=2, max=50, step=1, description="Krylov m:")
lan_kind = widgets.Dropdown(options=[("Real SPD", "real"),
                                         ("Complex HPD", "complex")], description="System:")
lan_run = widgets.Button(description="Run Lanczos", button_style="primary")
lan_out = widgets.Output()

def run_lanczos(_=None):
    with lan_out:
        clear_output(wait=True)
        n = lan_n.value
        m = min(lan_m.value, n)
        generator = hermitian_pd_problem if lan_kind.value == "complex" else spd_problem
        A, b, x_true = generator(n=n, condition=100, seed=17)
        projection = lanczos_tridiagonalize(A, b, m=m, reorthogonalize=True)
        V, T = projection.basis, projection.tridiagonal
        ortho_error = np.linalg.norm(V.conj().T @ V - np.eye(V.shape[1]))
        projection_error = np.linalg.norm(V.conj().T @ A @ V - T)
        print(f"Dimension={n}, requested m={m}, effective m={projection.iterations}")
        print(f"Orthogonality error={ortho_error:.3e}")
        print(f"Projection error={projection_error:.3e}")
        print(f"Ritz range=[{projection.ritz_values[0]:.4g}, "
              f"{projection.ritz_values[-1]:.4g}]")
        ritz_spectrum_plot(A, projection.ritz_values)
        ritz_residual_plot(A, V, T)
        gal = lanczos_galerkin(A, b, m=m, tol=1e-9, reorthogonalize=True)
        if gal.solution is None:
            print("Galerkin projection unavailable:", gal.message)
        else:
            residual = np.linalg.norm(b-A@gal.solution)
            error = np.linalg.norm(gal.solution-x_true)/np.linalg.norm(x_true)
            print(f"Galerkin converged={gal.converged} | "
                  f"true residual={residual:.3e} | relative error={error:.3e}")

lan_run.on_click(run_lanczos)
display(widgets.VBox([lan_n, lan_m, lan_kind, lan_run, lan_out]))

## Conclusions and interpretation

- Steepest Descent is a **gradient-method baseline**; its rate is sensitive
  to conditioning. CG leverages conjugacy and usually improves on it for SPD/HPD systems.
- For general nonsymmetric systems, GMRES and BiCGSTAB are appropriate;
  restarted GMRES trades memory for potentially slower convergence.
- The residual and forward error are **different accuracy measures**.
  Residuals should be recomputed from $b-Ax$ when checking convergence.
- Lanczos Ritz values describe spectral information; an $m<n$ Galerkin
  projection is not guaranteed to produce a converged solution.

**Reproducibility:** All experiments use fixed seeds. Runtime comparisons
are illustrative, especially in public Binder environments. The GMRES
callbacks compute additional true residuals, so their timings include
diagnostic overhead. Use the individual notebooks for preconditioner
comparisons and breakdown/stagnation experiments.